In [1]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
import torch.nn as nn
import torch.optim as optim

from helper_modules import phase1_preprocess_data, phase2_preprocess_data

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)


### Read in the (test/submission) data

In [2]:
df_test = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_test.csv'))
# df_test = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_test_partial.csv'))
print(df_test.shape)
display(df_test.head())

(35040, 5)


,Site,Timestamp_Local,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW
0,siteD,2020-01-01 00:00:00,24.70,0,4.8
1,siteD,2020-01-01 00:15:00,24.61,0,4.8
2,siteD,2020-01-01 00:30:00,24.53,0,4.8
3,siteD,2020-01-01 00:45:00,24.45,0,4.8
4,siteD,2020-01-01 01:00:00,24.36,0,4.8


### Stage 1 Predictions -- Classifying Demand_Response_Flag

In [3]:
# perform some preprocessing for phase 1
df_test['Demand_Response_Capacity_kW'] = np.nan  # Add the target column with NaN values
df_test['Demand_Response_Flag'] = np.nan  # Add the target column with NaN values
df = phase1_preprocess_data(df_test)

In [4]:
df.drop(columns=['Demand_Response_Flag'], inplace=True)

In [5]:
# Load the trained model from file
from net_architecture import Net    # Define the neural network architecture: Just need to load architecture defined in net_architecture.py
input_dim = df.shape[1]             # Number of features
num_classes = 3                     # Number of classes in target variable
model_loaded = Net(input_dim, num_classes)
model_loaded.load_state_dict(torch.load('./models/phase1_nn_model.pth'))

# load the scaler, which should be ColumnTransformer type
# scaler is a combination of PowerTransformer (Yeo-Johnson transformation) & StandardScaler
scaler = joblib.load('./models/phase1_scaler.pkl')
print(scaler)

ColumnTransformer(transformers=[('bin_pass', 'passthrough',
                                 ['is_daylight', 'Is_Weekend', 'Is_Summer',
                                  'Is_Winter', 'Is_Afternoon', 'Is_Evening']),
                                ('yj_std',
                                 Pipeline(steps=[('yeojohnson',
                                                  PowerTransformer(standardize=False)),
                                                 ('std', StandardScaler())]),
                                 ['Dry_Bulb_Temperature_C',
                                  'Global_Horizontal_Radiation_W/m2',
                                  'Building_Power_kW', 'Hour', 'Day', 'DOW',
                                  'Month', 'Weekday', 'Minute', 'Hour_sin',
                                  'Hour_cos', 'DOW_sin', 'DOW_cos', 'HDD18',
                                  'CDD22', 'TempC2', 'rad_sqrt', 'rad_log1p',
                                  'TempC_x_daylight', 'CDD22_x_daylight',


In [6]:
# Convert DataFrame to torch tensor
X = scaler.transform(df)
X = torch.tensor(X, dtype=torch.float32)

# Set model to evaluation mode
model_loaded.eval()
with torch.no_grad():
    outputs = model_loaded(X)
    predictions = torch.argmax(outputs, dim=1)

# convert 2 in predictions to -1
predictions = np.where(predictions == 2, -1, predictions)

# Calculate the distribution of predictions classes
unique, counts = np.unique(predictions, return_counts=True)
distribution_pred = dict(zip(unique, counts))
print(distribution_pred)

{np.int64(-1): np.int64(6041), np.int64(0): np.int64(22981), np.int64(1): np.int64(6018)}


In [7]:
# Prepare data for Stage 2 predictions
df_stage2 = df_test.copy(deep=True)
df_stage2['Demand_Response_Flag'] = predictions

### Stage 2 Predictions - Demand_Response_Capacity_kW

In [8]:
# perform some preprocessing for phase 2
df = phase2_preprocess_data(df_stage2)
df.drop(columns=['Demand_Response_Flag','Demand_Response_Capacity_kW'], inplace=True)
df.head()

,Dry_Bulb_Temperature_C,Global_Horizontal_Radiation_W/m2,Building_Power_kW,Hour,Day,DOW,Month,Weekday,Minute,Hour_sin,Hour_cos,DOW_sin,DOW_cos,HDD18,CDD22,TempC2,rad_sqrt,rad_log1p,is_daylight,TempC_x_daylight,CDD22_x_daylight,TempC_x_hour_sin,TempC_x_hour_cos,Is_Weekend,Is_Summer,Is_Winter,Is_Afternoon,Is_Evening
0,24.70,0,4.8,0,1,2,1,2,0,0.000000,1.000000,0.974928,-0.222521,0.0,2.70,610.0900,0.0,0.0,0,0.0,0.0,0.000000,24.700000,0,0,1,0,0
1,24.61,0,4.8,0,1,2,1,2,15,0.000000,1.000000,0.974928,-0.222521,0.0,2.61,605.6521,0.0,0.0,0,0.0,0.0,0.000000,24.610000,0,0,1,0,0
2,24.53,0,4.8,0,1,2,1,2,30,0.000000,1.000000,0.974928,-0.222521,0.0,2.53,601.7209,0.0,0.0,0,0.0,0.0,0.000000,24.530000,0,0,1,0,0
3,24.45,0,4.8,0,1,2,1,2,45,0.000000,1.000000,0.974928,-0.222521,0.0,2.45,597.8025,0.0,0.0,0,0.0,0.0,0.000000,24.450000,0,0,1,0,0
4,24.36,0,4.8,1,1,2,1,2,0,0.258819,0.965926,0.974928,-0.222521,0.0,2.36,593.4096,0.0,0.0,0,0.0,0.0,6.304832,23.529953,0,0,1,0,0


In [9]:
# Load models
clf_nz = joblib.load('./models/phase2_xgb_clf_nz_model.pkl')
reg_nz = joblib.load('./models/phase2_xgb_reg_nz_model.pkl')

# Prepare test features (X_test)
X_test = df.values

# Predict on real test set
p_nz = clf_nz.predict_proba(X_test)[:, 1]
mu_nz = reg_nz.predict(X_test)
y_pred_xgb = p_nz * mu_nz


In [10]:
df_submission = df_test[['Site', 'Timestamp_Local']].copy(deep=True)
df_submission['Demand_Response_Flag'] = predictions
df_submission['Demand_Response_Capacity_kW'] = y_pred_xgb

# manually enforce 0 for Demand_Response_Capacity_kW if Demand_Response_Flag = 0
df_submission.loc[df_submission['Demand_Response_Flag'] == 0, 'Demand_Response_Capacity_kW'] = 0

df_submission.to_csv('./submissions/submission.csv', index=False)
print("Submission file created: submission.csv in ./submissions folder")

Submission file created: submission.csv in ./submissions folder


In [11]:
# df_submission = df_test[['Site', 'Timestamp_Local']].copy(deep=True)
# df_submission['Demand_Response_Flag'] = predictions
# df_submission['Demand_Response_Capacity_kW'] = y_pred_xgb

# # manually enforce 0 for Demand_Response_Capacity_kW if Demand_Response_Flag = 0
# df_submission.loc[df_submission['Demand_Response_Flag'] == 0, 'Demand_Response_Capacity_kW'] = 0

# df_pred_partial = pd.read_csv(os.path.join('submissions', 'flextrack_phase1_test_partial_predictions.csv'))
# df_submission = pd.concat([df_submission, df_pred_partial], axis=0)
# df_submission = df_submission.sort_values(by=['Site', 'Timestamp_Local']).reset_index(drop=True)

# df_submission.to_csv('./submissions/submission.csv', index=False)
# print("Submission file created: submission.csv in ./submissions folder")

In [12]:
display(df_submission.head())
print('submission file shape:', df_submission.shape)
df_submission['Demand_Response_Flag'].value_counts()

,Site,Timestamp_Local,Demand_Response_Flag,Demand_Response_Capacity_kW
0,siteD,2020-01-01 00:00:00,0,0.0
1,siteD,2020-01-01 00:15:00,0,0.0
2,siteD,2020-01-01 00:30:00,0,0.0
3,siteD,2020-01-01 00:45:00,0,0.0
4,siteD,2020-01-01 01:00:00,0,0.0


submission file shape: (35040, 4)


Demand_Response_Flag
 0    22981
-1     6041
 1     6018
Name: count, dtype: int64